# Dolphi cookbook: evaluating a ticker

This notebook runs Dolphi end-to-end in mock mode — no API keys, no network.

You'll see the bull/bear debate, the pre-mortem falsifiers, the fragility scores that determine position sizing, and the falsifier-check loop.

Replace `mock=True` with a real config to run against live LLMs.

In [ ]:
from dolphi import evaluate

result = evaluate(symbols=["NVDA", "AMD", "TSM"], mock=True, top_k=5)
print(f"Decision ID: {result.decision_id}")
print(f"Symbols evaluated: {[a.symbol for a in result.allocations]}")

In [ ]:
import pandas as pd

ideas_df = pd.DataFrame([i.model_dump() for i in result.ranked_ideas])
ideas_df[["rank", "symbol", "score", "confidence", "thesis"]].head(10)

In [ ]:
sym = result.allocations[0].symbol
print(f"\n=== {sym} ===")
if sym in result.debate:
    print(f"\nDebate verdict: {result.debate[sym].winner.upper()}  "
          f"(conviction delta {result.debate[sym].conviction_delta:+.2f})")
    print(f"  {result.debate[sym].rationale}\n")
else:
    print("\nDebate verdict: (no judge verdict recorded in mock run)\n")
print("Falsifiers:")
for i, f in enumerate(result.falsifiers.get(sym, [])):
    print(f"  [{i}] {f.failure_mode}")
    print(f"      breaks: {f.breaks_assumption}")
    print(f"      watch:  {f.leading_indicator}  (within {f.horizon})")
    print(f"      probability: {f.probability:.2f}\n")

In [ ]:
def _color(frag):
    if frag <= 0.40:
        return "\U0001F7E2"
    elif frag <= 0.65:
        return "\U0001F7E1"
    return "\U0001F534"

frag_df = pd.DataFrame([
    {"symbol": s, "fragility": f, "indicator": _color(f)}
    for s, f in sorted(result.fragility.items(), key=lambda kv: kv[1])
])
frag_df

In [ ]:
from pathlib import Path
from IPython.display import SVG, Markdown, display

svg_path = Path("../docs/benchmarks/equity_curve.svg")
if svg_path.exists():
    display(SVG(svg_path.read_text()))
else:
    display(Markdown("*Walk-forward equity curve not generated yet. Run "
                     "`dolphi --backtest --mock-data` to produce it.*"))

In [ ]:
from dolphi import check_falsifiers

# Pretend a leading indicator fired for the top falsifier on the first symbol.
sym = result.allocations[0].symbol
fixture_path = Path("../tests/fixtures/sample_decision_log.jsonl")
adj = check_falsifiers(
    feedback={f"{sym}-0": "triggered"},
    jsonl_path=fixture_path,
)
print(f"Adjustment for {sym}: {adj.position_adjustments.get(sym, 0):+.0%}")
print(f"Triggered falsifiers: {len(adj.triggered_falsifiers)}")

## Run this on your own watchlist

To run against live LLMs with your own profile:

```python
from dolphi import evaluate, UserProfile

profile = UserProfile(
    total_savings=100_000,
    monthly_salary=8_000,
    risk_tolerance="Moderate",
    goal="growth",
    preferred_asset_classes=["stocks", "etfs"],
)

result = evaluate(
    symbols=["NVDA", "AMD", "TSM", "AAPL", "MSFT"],
    profile=profile,
    mock=False,  # uses your configured LLM provider
)
```

Or just use the CLI:

```bash
dolphi --new-profile --seed-symbols NVDA,AMD,TSM --top-k 5
dolphi --check  # weekly falsifier loop
```